# PRAXIS paper figures

### - code to generate all figures in the main body and supplemental sections of the publication
### - figures are generated with labels and descriptors in this notebook, then as a 'minimal' svg without labels in /exports to be marked and cleaned up for publication

## imports & setup

In [ ]:
%pip install pyfamsa scikit-learn
import os
os.makedirs('exports', exist_ok=True)
import pandas as pd
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
plt.style.use('data/praxis.mplstyle')
COLORS = {'glucose': '#5a63a6', 'xylose': '#ff4470', 'mannose': '#ffa600', 'accent': '#00a285'}
import numpy as np
import ternary
import colorsys
import matplotlib.colors as mcolors
from matplotlib.figure import Figure
from scipy.stats import linregress
import kinetics as K
from scipy.stats import pearsonr, spearmanr
from matplotlib.figure import Figure
import ternary
from pyfamsa import Aligner, Sequence
from sklearn.manifold import MDS
from scipy.spatial.distance import pdist, squareform
from matplotlib.patches import Circle
from random import randint, seed as rseed

In [ ]:
# connect enzyme engineering campaign DB
conn = sqlite3.connect("data/Bgl_spec_2026_final_database.db")
data = pd.read_sql("SELECT * FROM Bgl_spec_2026_final_assayed_sequences",conn)

In [ ]:
# shared setup - extract info fro DB
eps = 0.1

# a1/a2/a3 are log1p(raw rate); revert to raw rates
data['ga'] = np.expm1(data['a1']) + eps
data['xa'] = np.expm1(data['a2']) + eps
data['ma'] = np.expm1(data['a3']) + eps

# keep logged versions 
data['lga'], data['lxa'], data['lma'] = data['a1'], data['a2'], data['a3']

tot = data['ga'] + data['xa'] + data['ma']
data['gs'],  data['xs'],  data['ms']  = data['ga']/tot,     data['xa']/tot,     data['ma']/tot        # single-numerator spec
data['gs2'], data['xs2'], data['ms2'] = data['ga']**2/tot,  data['xa']**2/tot,  data['ma']**2/tot     # squared-numerator spec
data['x_shift'] = data['xa'] / data['ga']    # X/G
data['m_shift'] = data['ma'] / data['ga']    # M/G

# rounds: group by >20 min gaps in created_at (round 0 = seeds -> rounds 0..24)
mins = (pd.to_datetime(data['created_at']) - pd.to_datetime(data['created_at']).iloc[0]).dt.total_seconds() / 60
round_id, r = [0], 0
for i in range(1, len(mins)):
  if mins.iloc[i] - mins.iloc[i-1] > 20:
      r += 1
  round_id.append(r)
data['round'] = round_id

# per-agent frames: glucose / xylose / mannose optimizers
agents = [data[data['created_by'] == f'agent_a{i}'] for i in (1, 2, 3)]

# color palette
COLORS = {'glucose': '#5a63a6', 'xylose': '#ff4470', 'mannose': '#ffa600', 'accent': '#00a285'}

In [ ]:
import os
from matplotlib.figure import Figure
from matplotlib.transforms import Bbox
os.makedirs('exports', exist_ok=True)

def save_minimal(draw, name, fig, sharey=False):
    """Redraw `draw` on a bare Figure matching `fig`'s layout and save as SVG."""

    mfig = Figure(figsize=fig.get_size_inches())

    # Recover original grid geometry
    gs = fig.axes[0].get_subplotspec().get_gridspec()
    nrows, ncols = gs.nrows, gs.ncols

    axs = mfig.subplots(nrows, ncols, sharey=sharey, squeeze=False)

    draw(axs, minimal=True)

    for ax in axs.ravel():
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    sp = fig.subplotpars
    mfig.subplots_adjust(
        left=sp.left,
        right=sp.right,
        top=sp.top,
        bottom=sp.bottom,
        wspace=sp.wspace,
        hspace=sp.hspace,
    )

    w, h = fig.get_size_inches()
    mfig.savefig(
        f"exports/{name}.svg",
        bbox_inches=Bbox([[0, 0], [w, h]])
    )

# Fig 3b
## Plot campaign course as seqs over time

In [ ]:
# specificity trajectories 

seed = data[data['created_by'] == 'seed']
data['gs2_fc'] = data['gs2'] / seed['gs2'].max()
data['xs2_fc'] = data['xs2'] / seed['xs2'].max()
data['ms2_fc'] = data['ms2'] / seed['ms2'].max()

panels = [('Glucose', 'gs2_fc', COLORS['glucose']),
        ('Xylose',  'xs2_fc', COLORS['xylose']),
        ('Mannose', 'ms2_fc', COLORS['mannose'])]
rounds = range(25)  

def draw(axs, minimal=False): 
  for i, (ax, (sugar, col, color)) in enumerate(zip(np.asarray(axs).ravel(), panels)):
      y = data[col].clip(lower=0.1)
      best = np.maximum.accumulate([y[data['round'] == r].max() for r in rounds])
      ax.axhline(1.0, color='#999999', ls='--', lw=1, zorder=1)
      ax.plot(data['round'], y, 'o', color=color, markersize=6,
              markeredgecolor='white', markeredgewidth=0.6, alpha=0.85, zorder=2,
              label='Tested enzymes')                                   # circles
      ax.plot(rounds, best, '-', color=color, lw=2.2, zorder=3,
              label='Cumulative best')                                  # line
      ax.set_yscale('log')
      ax.set_ylim(0.08, None)
      if not minimal:
          ax.set_title(sugar, fontsize=12, fontweight='bold', color=color)
          if i == 0: 
              ax.legend(loc='upper right', fontsize=8, bbox_to_anchor=(1.0, .97))              

# full (displayed)  
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
draw(axs)
axs[0].set_ylabel('Specificity fold-change over parents')
fig.supxlabel('Round')
fig.tight_layout()  
plt.show()

# minimal (saved to exports/, not displayed)
save_minimal(draw, 'specificity_foldchange_over_parents', fig)

# Fig 3c
## Ternary plot of specificity trajectories

In [ ]:
objectives  = ['gs2', 'xs2', 'ms2']
titles      = ['Agent G', 'Agent X', 'Agent M']
tcolors     = [COLORS['glucose'], COLORS['xylose'], COLORS['mannose']]
act_thresh  = 1.0
pseudocount = 0.2

def best_trajectory(agent_df, objective):
    traj = []
    for rd in range(25):
        rdat  = agent_df[agent_df['round'] == rd]
        alive = rdat[(rdat['lga'] + rdat['lxa'] + rdat['lma']) > act_thresh]
        if len(alive):
            row = alive.loc[alive[objective].idxmax()]
            prof = [
                row['lxa'] + pseudocount,
                row['lga'] + pseudocount,
                row['lma'] + pseudocount,
            ]
            traj.append([p / sum(prof) for p in prof])
    return traj

def draw(axs, minimal=False):
    for i, ax in enumerate(np.asarray(axs).ravel()):
        color = tcolors[i]
        traj  = best_trajectory(agents[i], objectives[i])

        tax = ternary.TernaryAxesSubplot(ax=ax, scale=1.0)

        # Light gray triangle background
        for p in ax.patches:
            p.set_facecolor("#fafafa")

        # Gray gridlines
        tax.gridlines(
            multiple=0.2,
            color="#c9c9c9",
            linewidth=0.8,
            linestyle="-"
        )

        tax.boundary(
            linewidth=1.2,
            axes_colors={
                "b": "black",
                "l": "black",
                "r": "black"
            }
        )

        # Trajectory with white marker borders
        tax.plot(
            traj,
            linewidth=2,
            marker="o",
            markersize=7,
            color=color,
            markeredgecolor="white",
            markeredgewidth=1.2,
        )

        ax.axis("off")
        tax.clear_matplotlib_ticks()

        if not minimal:
            tax.right_corner_label("Xyl", fontsize=10)
            tax.top_corner_label("Glu", fontsize=10)
            tax.left_corner_label("Man", fontsize=10)
            tax.set_title(
                titles[i],
                fontsize=12,
                fontweight="bold",
                color=color,
                pad=30,
            )

# full (displayed)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
draw(axs)
fig.tight_layout()
plt.subplots_adjust(wspace=0.15)
plt.show()

# minimal (saved to exports/, not displayed)
save_minimal(draw, 'specificity2_ternary_trajectories', fig)

# Fig 3d
## Specificity trajectories on MDS sequence space

In [ ]:
# fragment codes -> align -> background -> trajectories -> embed 
blocks = tuple(zip(*[eval(s) for s in data.iloc[:6]['segmented_sequence']]))   # 8 positions x 6 parents
ss = [eval(s) for s in data['segmented_sequence']]
data['block_seq'] = [tuple(blocks[i].index(bl) for i, bl in enumerate(s)) for s in ss]

aligned_blocks = [] 
for block in blocks:                                                            # MSA-align each position's 6 parent fragments
  aln = Aligner().align([Sequence(f'seq{i}'.encode(), frag.encode()) for i, frag in enumerate(block)])
  aligned_blocks.append([s.sequence.decode() for s in aln])
data['aligned_seq'] = [''.join(aligned_blocks[i][p] for i, p in enumerate(bs)) for bs in data['block_seq']]

rseed(0)
random_chimeras = [[randint(0, 5) for _ in range(8)] for _ in range(1000)]      # random background
random_aligned  = [''.join(aligned_blocks[i][p] for i, p in enumerate(rc)) for rc in random_chimeras]

objectives = ['gs2', 'xs2', 'ms2']
trajectories = []   
for i, agent in enumerate(agents):                                              # best-per-round path, from parent 1
  traj = [0]
  for rd in range(1, 25):   
      alive = agent[(agent['round'] == rd) & ((agent['ga'] + agent['xa'] + agent['ma']) > 10)]
      if len(alive):
          traj.append(alive[objectives[i]].idxmax())
  trajectories.append(traj)

seqs = list(data['aligned_seq']) + random_aligned
alphabet = {c: k for k, c in enumerate(sorted(set(''.join(seqs))))}
X = np.array([[alphabet[c] for c in s] for s in seqs], dtype=np.int16)
D = squareform(pdist(X, metric='hamming'))
coords = MDS(n_components=2, dissimilarity='precomputed',
           random_state=0, normalized_stress='auto').fit_transform(D)

# ---- plot (circular bound, no axes) ----
tcolors = [COLORS['glucose'], COLORS['xylose'], COLORS['mannose']]
titles  = ['Agent G', 'Agent X', 'Agent M']
cx = (coords[:, 0].min() + coords[:, 0].max()) / 2
cy = (coords[:, 1].min() + coords[:, 1].max()) / 2
R  = np.hypot(coords[:, 0] - cx, coords[:, 1] - cy).max() * 1.01

def draw(axs, minimal=False): 
  axs = np.asarray(axs).ravel()
  for i, ax in enumerate(axs):
      ax.plot(coords[:, 0], coords[:, 1], '.', color='#cccccc', alpha=0.5, zorder=1)
      ax.plot(coords[trajectories[i], 0], coords[trajectories[i], 1], 'o-',
              color=tcolors[i], markeredgecolor='white', markeredgewidth=0.5, zorder=3,linewidth=2.5)
      ax.plot(coords[0, 0], coords[0, 1], '*', color=COLORS['accent'], markersize=18, markeredgewidth=0.8, zorder=4)
      ax.add_patch(Circle((cx, cy), R, fill=False, edgecolor='black', linewidth=1.0, zorder=2))
      ax.set_aspect('equal')
      ax.set_xlim(cx - R*1.04, cx + R*1.04)
      ax.set_ylim(cy - R*1.04, cy + R*1.04)
      ax.set_xticks([]); ax.set_yticks([])
      for s in ax.spines.values():
          s.set_visible(False)
      if not minimal:
          ax.set_title(titles[i], fontsize=12, fontweight='bold', color=tcolors[i])

# full (displayed)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
draw(axs)
fig.tight_layout()
plt.show()

# minimal (saved to exports/, not displayed)
save_minimal(draw, 'mds_trajectories', fig)

# Fig 3e and F
## Specificity shifts versus activities of tested variants

In [ ]:
panels = [('Xylose',  'xa', 'x_shift', COLORS['xylose']),
          ('Mannose', 'ma', 'm_shift', COLORS['mannose'])]
thresh = 0.1   # eps

def draw(axs, minimal=False):
    axs = np.asarray(axs).ravel()
    for ax, (sugar, act_col, shift_col, color) in zip(axs, panels):
        sub     = data[data[act_col] > thresh]
        is_seed = sub['round'] == 0

        ax.loglog(
            sub.loc[~is_seed, act_col],
            sub.loc[~is_seed, shift_col],
            'o',
            color=color,
            markersize=8,
            markeredgecolor='white',
            markeredgewidth=0.5,
            alpha=1,
            zorder=2
        )

        ax.loglog(
            sub.loc[is_seed, act_col],
            sub.loc[is_seed, shift_col],
            'o',
            color=COLORS['accent'],
            markersize=8,
            markeredgecolor='white',
            markeredgewidth=0.5,
            zorder=3,
            label='Seed'
        )

        ax.set_box_aspect(1)

        # Gridlines
        ax.grid(
            True,
            which='major',
            color='#bfbfbf',
            linewidth=0.8,
            linestyle='-',
            alpha=0.8
        )
        ax.grid(
            True,
            which='minor',
            color='#dddddd',
            linewidth=0.5,
            linestyle='-',
            alpha=0.6
        )

        if not minimal:
            ax.set_title(sugar, fontsize=12, fontweight='bold', color=color)
            ax.set_xlabel(f'{sugar} activity')

    if not minimal:
        axs[0].set_ylabel('Specificity shift (target / glucose)')
        axs[0].legend(loc='best', fontsize=8)

# full (displayed)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
draw(axs)
fig.tight_layout()
plt.show()

# minimal (saved to exports/, not displayed)
save_minimal(draw, 'specificity_shift_vs_activity', fig)

# Fig 3g
## Xylose versus mannose shifts

In [ ]:
thresh = 0.1
AXIS_LW, TICK_LEN, TICK_W = 1.4, 6, 1.4      # spine thickness, tick length, tick width

def draw(axs, minimal=False): 
  axs = np.asarray(axs).ravel()[0]
  sub     = data[(data['xa'] > thresh) | (data['ma'] > thresh)]
  is_seed = sub['round'] == 0

  ax.loglog(sub.loc[~is_seed, 'x_shift'], sub.loc[~is_seed, 'm_shift'], 'o',
            color='#888888', markersize=11, markeredgecolor='white',
            markeredgewidth=0.5, alpha=0.6, zorder=2)
  ax.loglog(sub.loc[is_seed, 'x_shift'], sub.loc[is_seed, 'm_shift'], 'o',
            color=COLORS['accent'], markersize=11, markeredgecolor='white',
            markeredgewidth=1, zorder=3, label='Seed')

  # equal, square, log limits
  vals = np.concatenate([sub['x_shift'].values, sub['m_shift'].values])
  vals = vals[np.isfinite(vals) & (vals > 0)]
  lo, hi = 10**np.floor(np.log10(vals.min())), 10**np.ceil(np.log10(vals.max()))
  ax.set_xlim(lo, hi)
  ax.set_ylim(lo, hi)
  ax.set_box_aspect(1)

  # thicker axes + bigger ticks (applied in both full and minimal)
  for s in ax.spines.values(): 
      s.set_linewidth(AXIS_LW)
  ax.tick_params(which='major', width=TICK_W, length=TICK_LEN)
  ax.tick_params(which='minor', width=TICK_W * 0.75, length=TICK_LEN * 0.6)

  if not minimal: 
      ax.set_xlabel('Xylose shift (X/G)')
      ax.set_ylabel('Mannose shift (M/G)')
      ax.legend(loc='best', fontsize=8)

# full (displayed)
fig, ax = plt.subplots(figsize=(5, 5))
draw(ax)
fig.tight_layout()
plt.show()

# minimal (saved to exports/, not displayed)
save_minimal(draw, 'xylose_vs_mannose_shift', fig)

# Fig 3h
## Campaign timeline

In [ ]:
  import matplotlib.dates as mdates
  
  data['created_at'] = pd.to_datetime(data['created_at'])
  gaps = data['created_at'].diff().dt.total_seconds() / 60      # minutes between experiments
  data['round'] = (gaps > 20).cumsum()                          # 0-indexed rounds
  round_times = data.groupby('round')['created_at'].min()
  start, stop = round_times.iloc[0], data['created_at'].max()

  LINE_LW, CIRCLE_LW, AXIS_LW = 5.0, 3.0, 3.0                   # tune thickness here
  
  def draw(axs, minimal=False):
      axs = np.asarray(axs).ravel()[0]
      ax.plot([round_times.iloc[0], round_times.iloc[-1]], [0, 0], '-',
              color=COLORS['accent'], linewidth=LINE_LW, zorder=1)
      ax.scatter(round_times, [0] * len(round_times), s=350,
                 facecolors='white', edgecolors='black', linewidth=CIRCLE_LW, zorder=3)
      ax.set_ylim(-0.45, 0.45); ax.set_yticks([])
      for sp in ['left', 'right', 'top']:
          ax.spines[sp].set_visible(False)
      ax.spines['bottom'].set_linewidth(AXIS_LW)
      ax.tick_params(axis='x', width=AXIS_LW)
      if minimal:
          ax.set_xticks([])                                     # no ticks, numbers, or text
      ax.spines['bottom'].set_linewidth(AXIS_LW)
      ax.tick_params(axis='x', width=AXIS_LW)
      ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))     # tick MARKS in both full + minimal
      if not minimal:
          for rn, ts in round_times.items():
              ax.text(ts, 0, str(rn), ha='center', va='center', fontsize=8, zorder=4)
          ax.annotate(f"Start\n{start:%b %d}", xy=(start, 0), xytext=(0, 35),
                      textcoords='offset points', ha='center', arrowprops=dict(arrowstyle='-', linewidth=0.8))
          ax.annotate(f"Stop\n{stop:%b %d}", xy=(stop, 0), xytext=(0, -45),
                      textcoords='offset points', ha='center', arrowprops=dict(arrowstyle='-', linewidth=0.8))
          for m in pd.date_range(start=start.normalize() + pd.offsets.MonthBegin(1),
                                 end=stop.normalize(), freq='MS'):
              ax.axvline(m, linestyle='--', linewidth=0.8, color='#999999')
              ax.text(m, 0.17, m.strftime('%B %Y'), ha='center', va='bottom', fontsize=9)
          ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
          ax.set_xlabel('Date')
  
  # full (displayed + saved)
  fig, ax = plt.subplots(figsize=(12, 2.5))
  draw(ax)
  fig.tight_layout()  
  fig.savefig('exports/round_timeline.svg', bbox_inches='tight')
  plt.show()
  
  # minimal (saved, not displayed)
  save_minimal(draw, 'round_timeline_minimal', fig)

# Fig 4a and b
## Kinetics plots of all parents and characterized variants + kinetic parameters

In [ ]:
import os
from matplotlib.figure import Figure
import kinetics as K

KDIR = "data/kinetics"
N_SLOPE = 3
SUBS = ["glu", "xyl", "man"]
SUB_LABELS = {"glu": "Glucose", "xyl": "Xylose", "man": "Mannose"}
SUB_COLORS = {"glu": COLORS["glucose"], "xyl": COLORS["xylose"], "man": COLORS["mannose"]}
FOLD_FLOOR, FOLD_CEIL = 0.65, 5.0
CONSTIT = {
    "G1": [2, 4],
    "G2": [2, 6],
    "G6": [2, 5, 6],
    "G9": [2, 4, 6],
    "G15": [1, 3, 5],
    "X1": [2, 5],
    "X3": [2, 4, 6],
    "X7": [3, 4, 6],
    "X8": [4, 5],
    "M7": [4, 6],
    "M13": [2, 4],
    "M21": [1, 2, 4, 5],
}


def _fit(c, y, s, e):
    se = K.estimate_initial_slope(c, y, N_SLOPE)
    fit = K.fit_substrate_inhibition(c, y, s, e)
    if fit and se and se["slope"] > 0:
        fit = K.fit_si_constrained(c, y, s, e, se["slope"], se["slope_SE"]) or fit
    return fit, se


results = {}

# ---- parents ----
PARENTS = ["p1", "p2", "p3", "p4", "p5", "p6"]
ENZ_P = {"glu": 10, "xyl": 100, "man": 100}
TR_P = {"glu": None, "xyl": (1.0, 2.5), "man": (1.0, 2.5)}
GAIN = {"glu": "glucose", "xyl": "xylose", "man": "mannose"}
bkp = K.load_background_raw(f"{KDIR}/parents")
for sub in SUBS:
    sl, ic, _ = K.load_standard_curve_gain(f"{KDIR}/parents/std_curve.xlsx", GAIN[sub])
    e = ENZ_P[sub] / 1000.0
    tr = TR_P[sub]
    trs = (tr[0] * 3600, tr[1] * 3600) if tr else None
    for p in PARENTS:
        d = K.load_compiled_data(f"{KDIR}/parents/{p}_{sub}.xlsx")
        v = K.calculate_v0_for_compiled(d, sl, ic, time_range=trs, bkg_df=bkp.get(sub))
        c, y, s = (
            v["concentration"].to_numpy(float),
            v["v0_uM_per_s"].to_numpy(float),
            v["v0_sem_uM_per_s"].to_numpy(float),
        )
        fit, se = _fit(c, y, s, e)
        results.setdefault(p, {})[sub] = {"conc": c, "v0": y, "enzyme_uM": e, "fit": fit, "slope_est": se}

# ---- variants (alias-named; round = digits in the name) ----
VARIANTS = ["G1", "G2", "G6", "G9", "G15", "X1", "X3", "X7", "X8", "M7", "M13", "M21"]
VARIANTS.sort(key=lambda n: (int(n[1:]), n))
slv, icv, _ = K.load_standard_curve_extended(f"{KDIR}/variants/std_curve.xlsx")
tdf, tov, ecs = K.load_time_ranges(f"{KDIR}/variants/time_ranges.csv")
bkv = K.load_background_raw(f"{KDIR}/variants")
for enz in VARIANTS:
    lo = enz.lower()
    for sub in SUBS:
        fp = f"{KDIR}/variants/{enz}_{sub}.xlsx"
        nM = ecs.get((lo, sub))
        if not os.path.exists(fp) or nM is None:
            continue
        eu = nM / 1000.0
        tr = K.get_time_range(lo, sub, None, tdf, tov)
        ctr = {k[2]: v for k, v in tov.items() if k[0] == lo and k[1] == sub}
        d = K.load_compiled_data(fp)
        v = K.calculate_v0_for_compiled(
            d, slv, icv, time_range=tr, time_range_by_conc=ctr, bkg_df=bkv.get(sub)
        )
        c, y, s = (
            v["concentration"].to_numpy(float),
            v["v0_uM_per_s"].to_numpy(float),
            v["v0_sem_uM_per_s"].to_numpy(float),
        )
        fit, se = _fit(c, y, s, eu)
        results.setdefault(enz, {})[sub] = {"conc": c, "v0": y, "enzyme_uM": eu, "fit": fit, "slope_est": se}

COLUMNS = [(f"P{i}", f"p{i}") for i in range(1, 7)] + [(v, v) for v in VARIANTS]


# ---- kinetic-parameter outputs ----
def _slope_kk(r):
    se = (r or {}).get("slope_est")
    return (se["slope"] / r["enzyme_uM"]) * 1e6 if (se and se["slope"] > 0) else np.nan


def _slope_kk_se(r):
    se = (r or {}).get("slope_est")
    return (
        (se["slope_SE"] / r["enzyme_uM"]) * 1e6
        if (se and np.isfinite(se.get("slope_SE", np.nan)))
        else np.nan
    )


def _slope_r2(r):
    """R2 of the initial-slope linear fit that kcat/Km comes from -- not the MM fit."""
    se = (r or {}).get("slope_est")
    return se.get("R2_linear", np.nan) if se else np.nan


# (a) parent parameter table (written to the xlsx below)
prows = []
for i in range(1, 7):
    for sub in SUBS:
        r = results[f"p{i}"][sub]
        fit = r["fit"]
        prows.append(
            {
                "Protein": f"P{i}",
                "Substrate": SUB_LABELS[sub],
                "kcat/Km (M⁻¹s⁻¹)": _slope_kk(r),
                "kcat/Km SE": _slope_kk_se(r),
                "kcat/Km R2 (linear)": _slope_r2(r),
                "Vmax (µM/s)": fit["Vmax_uM_s"] if fit else np.nan,
                "Vmax SE": fit["Vmax_SE"] if fit else np.nan,
                "Km (µM)": fit["Km_uM"] if fit else np.nan,
                "Km SE": fit["Km_SE"] if fit else np.nan,
                "Ki (µM)": fit["Ki_uM"] if fit else np.nan,
                "Ki SE": fit["Ki_SE"] if fit else np.nan,
                "kcat (s⁻¹)": fit["kcat_s"] if fit else np.nan,
                "kcat SE": fit["kcat_SE"] if fit else np.nan,
                "R2 (MM fit)": fit["R2"] if fit else np.nan,
            }
        )
# (b) kcat_km.csv — flat kcat/Km for all enzymes; read by the figure cells below
pd.DataFrame(
    [
        {
            "Enzyme": disp,
            **{
                f"kcat/Km {SUB_LABELS[s]}": (
                    _slope_kk(results[key][s]) if results.get(key, {}).get(s) else np.nan
                )
                for s in SUBS
            },
        }
        for disp, key in COLUMNS
    ]
).to_csv("exports/kcat_km.csv", index=False)

# (c) kinetic_parameters.xlsx — parents get the full parameter set, variants
#     get kcat/Km only (their Vmax/Km are not separately identifiable here).
_pfull = pd.DataFrame(prows)

_vrows = []
for label, key in COLUMNS[6:]:
    row = {"Enzyme": label}
    for sub in SUBS:
        r = results.get(key, {}).get(sub)
        row[f"kcat/Km {SUB_LABELS[sub]} (M⁻¹s⁻¹)"] = _slope_kk(r)
        row[f"kcat/Km {SUB_LABELS[sub]} SE"] = _slope_kk_se(r)
        row[f"kcat/Km {SUB_LABELS[sub]} R2 (linear)"] = _slope_r2(r)
    _vrows.append(row)
_vkk = pd.DataFrame(_vrows)

with pd.ExcelWriter("exports/kinetic_parameters.xlsx") as _xw:
    _pfull.to_excel(_xw, sheet_name="parents", index=False)
    _vkk.to_excel(_xw, sheet_name="variants", index=False)

# ---- printed summary: kcat/Km for every enzyme ----
_summary = pd.DataFrame(
    [
        {"Enzyme": disp, **{SUB_LABELS[s]: _slope_kk(results.get(key, {}).get(s)) for s in SUBS}}
        for disp, key in COLUMNS
    ]
).set_index("Enzyme")
print("kcat/Km (M⁻¹s⁻¹)")
print(_summary.round(1).to_string(na_rep="n.d."))

# ---- fold-Δ references ----
PKK = {(i, s): _slope_kk(results[f"p{i}"][s]) for i in range(1, 7) for s in SUBS}


def parent_fractions(ids):
    per = {s: [] for s in SUBS}
    for pid in ids:
        clean = {
            s: (
                PKK.get((pid, s), 0)
                if np.isfinite(PKK.get((pid, s), np.nan)) and PKK.get((pid, s), 0) > 0
                else 0.0
            )
            for s in SUBS
        }
        tot = sum(clean.values())
        if tot <= 0:
            continue
        for s in SUBS:
            per[s].append(clean[s] / tot)
    return {s: (float(np.mean(per[s])) if per[s] else np.nan) for s in SUBS}


all_parents_frac = parent_fractions([1, 2, 3, 4, 5, 6])


def draw_fold(ax, enz, minimal, showy):
    kk = []
    for sub in SUBS:
        r = results.get(enz, {}).get(sub, {})
        se = r.get("slope_est")
        kk.append(
            (se["slope"] / r["enzyme_uM"]) * 1e6
            if (se and np.isfinite(se.get("slope", np.nan)) and se["slope"] > 0)
            else np.nan
        )
    tot = np.nansum([v for v in kk if np.isfinite(v) and v > 0])
    fr = [v / tot if np.isfinite(v) and v > 0 and tot > 0 else np.nan for v in kk]
    cf = parent_fractions(CONSTIT[enz])
    x = np.arange(3)
    for k, sub in enumerate(SUBS):
        f, fcn = fr[k], cf.get(sub, np.nan)
        fc = (f / fcn) if (np.isfinite(f) and f > 0 and np.isfinite(fcn) and fcn > 0) else np.nan
        if np.isfinite(fc) and fc > 0:
            ax.bar(
                x[k],
                min(fc, FOLD_CEIL * 0.98) - FOLD_FLOOR,
                bottom=FOLD_FLOOR,
                width=0.78,
                facecolor="none",
                edgecolor=SUB_COLORS[sub],
                linewidth=1.1,
                zorder=2,
            )
        elif not minimal:
            ax.text(
                x[k],
                FOLD_FLOOR * 1.12,
                "n.d.",
                ha="center",
                va="bottom",
                rotation=90,
                fontsize=5,
                color="#555",
            )
        fa = all_parents_frac.get(sub, np.nan)
        if np.isfinite(fcn) and fcn > 0 and np.isfinite(fa) and fa > 0:
            ax.scatter(
                x[k],
                min(max(fa / fcn, FOLD_FLOOR), FOLD_CEIL),
                s=10,
                facecolor=SUB_COLORS[sub],
                edgecolor="none",
                zorder=4,
            )
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, zorder=3)
    ax.set_yscale("log")
    ax.set_ylim(FOLD_FLOOR, FOLD_CEIL)
    ax.set_xlim(-0.6, 2.6)
    ax.set_box_aspect(1)
    ax.set_xticks(x)
    ax.set_yticks([0.75, 1, 2, 5])
    ax.set_yticks([], minor=True)
    if minimal:
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    else:
        ax.set_xticklabels(["G", "X", "M"], fontsize=6)
        if showy:
            ax.set_yticklabels([f"{t:g}×" for t in [0.75, 1, 2, 5]], fontsize=6)
        else:
            ax.set_yticklabels([])


def draw(axes, minimal=False):
    for r, sub in enumerate(SUBS):
        for c, (label, key) in enumerate(COLUMNS):
            ax = axes[r, c]
            color = SUB_COLORS[sub]
            res = results.get(key, {}).get(sub)
            ax.set_box_aspect(1)
            ax.set_xticks([])
            ax.set_yticks([])
            if res is not None and res["conc"].size > 0:
                conc, eu = res["conc"], res["enzyme_uM"]
                v0 = res["v0"] / eu
                popt = res["fit"]["popt"] if res.get("fit") else None
                ax.plot(conc, v0, "o", color=color, markersize=2.5, markeredgecolor="none", alpha=0.9)
                if popt is not None:
                    cs = np.linspace(0, conc.max() * 1.05, 300)
                    ax.plot(cs, K.substrate_inhibition(cs, *popt) / eu, color=color, linewidth=1.2)
                ax.set_xlim(left=0)
                ax.set_ylim(bottom=0)
            if not minimal:
                if r == 0:
                    ax.set_title(label, fontsize=8, fontweight="bold")
                if c == 0:
                    ax.set_ylabel(SUB_LABELS[sub], fontsize=9, fontweight="bold", color=color)
    for c, (label, key) in enumerate(COLUMNS):
        ax = axes[3, c]
        ax.set_box_aspect(1)
        if key.startswith("p"):
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        draw_fold(ax, key, minimal, showy=(c == 6))
        if not minimal and c == 6:
            ax.set_ylabel("fold Δ", fontsize=9, fontweight="bold")


fig, axes = plt.subplots(4, 18, figsize=(18, 4.5))
draw(axes, minimal=False)
fig.tight_layout()
plt.show()

mfig = Figure(figsize=(18, 4.5))
draw(mfig.subplots(4, 18), minimal=True)
mfig.tight_layout()
mfig.savefig("exports/mm_grid_fold_minimal.svg")

# Fig 4c and d
## Activity vs specificity (Xylose & Mannose)

In [ ]:
kk = pd.read_csv('exports/kcat_km.csv')
kk['X/G'] = kk['kcat/Km Xylose']  / kk['kcat/Km Glucose']
kk['M/G'] = kk['kcat/Km Mannose'] / kk['kcat/Km Glucose']

par = kk[kk['Enzyme'].str.startswith('P')]
best_x_act, best_m_act = par['kcat/Km Xylose'].max(), par['kcat/Km Mannose'].max()
best_xg,    best_mg    = par['X/G'].max(),           par['M/G'].max()
kk['X activity fold'] = kk['kcat/Km Xylose']  / best_x_act
kk['M activity fold'] = kk['kcat/Km Mannose'] / best_m_act
kk['X/G fold'] = kk['X/G'] / best_xg
kk['M/G fold'] = kk['M/G'] / best_mg

panels = [
  {'title': 'Xylose',  'act': 'X activity fold', 'spec': 'X/G fold',
   'xlabel': 'X activity fold over best parent', 'ylabel': 'X/G fold over best parent', 'color': COLORS['xylose']},
  {'title': 'Mannose', 'act': 'M activity fold', 'spec': 'M/G fold',
   'xlabel': 'M activity fold over best parent', 'ylabel': 'M/G fold over best parent', 'color': COLORS['mannose']},
]

BOX_LW = 1.4
def draw(axs, minimal=False):
  axs = np.asarray(axs).ravel()
  for ax, p in zip(axs, panels):
      d = kk.dropna(subset=[p['act'], p['spec']]).copy()
      is_parent = d['Enzyme'].str.startswith('P')
      ax.scatter(d.loc[~is_parent, p['act']], d.loc[~is_parent, p['spec']], s=75,
                 color=p['color'], alpha=0.62, edgecolor='black', linewidth=0.5, label='Designed', zorder=2)
      ax.scatter(d.loc[is_parent, p['act']], d.loc[is_parent, p['spec']], s=85,
                 color=p['color'], alpha=1.0, edgecolor='black', linewidth=0.9, marker='s', label='Parent', zorder=3)
      ax.axvline(1, color='black', linestyle=':',  linewidth=1, zorder=1)
      ax.axhline(1, color='black', linestyle='--', linewidth=1, zorder=1)
      ax.set_xscale('log'); ax.set_yscale('log'); ax.set_box_aspect(1)
      for spine in ax.spines.values():
          spine.set_linewidth(BOX_LW)
      # two-tier gridlines (major + minor)
      ax.grid(True, which='major', color='#bfbfbf', linewidth=0.8, linestyle='-', alpha=0.8)
      ax.grid(True, which='minor', color='#dddddd', linewidth=0.5, linestyle='-', alpha=0.6)
      ax.set_axisbelow(True)
      if not minimal:
          ax.set_xlabel(p['xlabel']); ax.set_ylabel(p['ylabel'])
          ax.set_title(p['title'], fontsize=12, fontweight='bold', color=p['color'])
  if not minimal:
      axs[0].legend(loc='best', fontsize=8)

fig, axs = plt.subplots(1, 2, figsize=(9.5, 4.6))
draw(axs) 
fig.tight_layout()
plt.show()

save_minimal(draw, 'activity_vs_specificity', fig)

# Fig 4e
## kcat/Km bar charts per substrate

In [ ]:
kcatkm_df = pd.read_csv('exports/kcat_km.csv')
BASE = 0.1   # positive baseline so log-scale bars don't run to log(0) = -inf

activity_cols = [('kcat/Km Glucose', 'Glucose', COLORS['glucose']),
               ('kcat/Km Xylose',  'Xylose',  COLORS['xylose']),
               ('kcat/Km Mannose', 'Mannose', COLORS['mannose'])]
BOX_LW = 1.4
def draw(axs, minimal=False):
  axs = np.atleast_1d(axs)  
  for ax, (col, title, color) in zip(axs, activity_cols):
      pdd = kcatkm_df[['Enzyme', col]].dropna().sort_values(col, ascending=False)
      is_parent = pdd['Enzyme'].str.startswith('P')
      ax.set_yscale('log')
      bars = ax.bar(pdd['Enzyme'], pdd[col] - BASE, bottom=BASE,
                    color=color, edgecolor='black', linewidth=0.5)
      for spine in ax.spines.values():
          spine.set_linewidth(BOX_LW)
      ax.set_ylim(bottom=BASE)
      for bar, ip in zip(bars, is_parent):
          bar.set_alpha(1.0 if ip else 0.55)
          bar.set_linewidth(0.8 if ip else 0.4)
      ax.set_axisbelow(True)
      ax.grid(axis='y', which='both', alpha=0.25)
      if not minimal:
          ax.set_title(title, fontsize=11, fontweight='bold', color=color)
          ax.set_xlabel('Enzyme')
          ax.tick_params(axis='x', rotation=90)
  if not minimal: 
      axs[0].set_ylabel(r'$k_{cat}/K_M$')
      parent_patch = plt.Rectangle((0, 0), 1, 1, color='gray', alpha=1.0,  ec='black', label='Parent')
      design_patch = plt.Rectangle((0, 0), 1, 1, color='gray', alpha=0.55, ec='black', label='Designed')
      axs[-1].legend(handles=[parent_patch, design_patch], frameon=False, loc='best')

fig, axs = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
draw(axs)
fig.tight_layout()
plt.show()

save_minimal(draw, 'kcatkm_bars_by_substrate', fig, sharey=True)

# Fig 4f
## Round checkpoint predictions on 1000 random enzymes

In [ ]:
import os, re, glob
from scipy.stats import gaussian_kde

# round sweep
_CANDS = ['../agent/scoring/round_sweep',
        'agent/scoring/round_sweep']

ROUND_DIR = next((p for p in _CANDS if os.path.isdir(p)), _CANDS[0])

def _round_id(fname):
  """Pull the round index out of a filename (round_24.csv, predictions_round24.csv, ...)."""
  m = re.search(r'round[_-]?(\d+)', fname, re.I)
  if m:
      return int(m.group(1))
  nums = re.findall(r'\d+', os.path.basename(fname))
  return int(nums[-1]) if nums else None

def load_round_sweep(round_dir=ROUND_DIR):
  if not os.path.isdir(round_dir):
      raise FileNotFoundError(f"round_sweep dir not found: {round_dir}")
  sweep = {}
  for f in sorted(glob.glob(os.path.join(round_dir, '*.csv'))):
      r = _round_id(f)
      if r is None:
          continue
      sweep[r] = pd.read_csv(f)          # expects columns: chimera, sequence, a1, a2, a3
  return sweep

sweep = load_round_sweep()
print(f"loaded {len(sweep)} rounds from {ROUND_DIR}: {sorted(sweep)}")

# --- Round sweep: 3x6 grid of smoothed activity histograms (KDE) --
from matplotlib.ticker import MultipleLocator

ROUNDS = [0, 4, 9, 14, 19, 24]
ACTS   = ["a1", "a2", "a3"]

# one color per activity row
ROW_COLORS = [COLORS["glucose"], COLORS["xylose"], COLORS["mannose"]]


def draw_round_sweep(axes, minimal=False):
    axes = np.asarray(axes).reshape(3, 6)

    for i, act in enumerate(ACTS):

        present = [r for r in ROUNDS if r in sweep]

        vals_all = np.concatenate([
            sweep[r][act].dropna().values
            for r in present
            if len(sweep[r][act].dropna()) > 0
        ])

        lo, hi = np.nanpercentile(vals_all, [0.5, 99.5])
        grid = np.linspace(lo, hi, 256)

        for j, r in enumerate(ROUNDS):

            ax = axes[i, j]

            if not minimal:
                for sp in ("top", "right", "left"):
                    ax.spines[sp].set_visible(False)

                ax.set_yticks([])
                ax.spines["bottom"].set_linewidth(1.6)
                ax.tick_params(axis="x", width=1.6, length=5)
                # Show every integer tick
                ax.xaxis.set_major_locator(MultipleLocator(1))

            if r not in sweep:
                if not minimal:
                    ax.text(
                        0.5, 0.5, "no data",
                        transform=ax.transAxes,
                        ha="center", va="center",
                        color="0.6", fontsize=8
                    )
                continue

            v = sweep[r][act].dropna().values

            # KDE requires at least two distinct observations
            if len(v) < 2 or np.all(v == v[0]):
                if len(v):
                    ax.axvline(v[0], color=ROW_COLORS[i], lw=2.8)
                continue

            dens = gaussian_kde(v)(grid)

            ax.plot(
                grid,
                dens,
                color=ROW_COLORS[i],
                lw=2.8,
                solid_capstyle="round",
            )

            if not minimal:
                if i == 0:
                    ax.set_title(f"Round {r}", fontsize=10)

                if j == 0:
                    ax.set_ylabel(
                        act,
                        rotation=0,
                        ha="right",
                        va="center",
                        fontsize=11,
                    )


fig, axes = plt.subplots(
    3, 6,
    figsize=(13, 5.5),
    sharex="row",
    sharey="row",
    constrained_layout=True,
)

draw_round_sweep(axes)

fig.supxlabel("predicted activity")

fig.savefig(
    "exports/round_sweep_activity_hist.png",
    dpi=300,
    bbox_inches="tight",
)

save_minimal(
    draw_round_sweep,
    "round_sweep_activity_hist",
    fig,
    sharey=False,
)

plt.show()

# Fig 4g
## Fragment analysis using final model checkpoint

In [ ]:
# cell(p, i) = mean(S | parent p at block position i) - mean(S),  S = (Xyl+Man)/2 - Glu (log1p space)
N_POS, N_PAR = 8, 6
_r24 = sweep[24] if (('sweep' in dir()) and 24 in sweep) else pd.read_csv(
  '../agent/scoring/round_sweep/predictions_round_24.csv', dtype={'chimera': str})
_r24 = _r24.astype({'chimera': str})

M = np.array([[int(c) for c in code] for code in _r24['chimera']], dtype=np.int8)   # (N,8) parent ids 1..6
a1, a2, a3 = _r24['a1'].to_numpy(), _r24['a2'].to_numpy(), _r24['a3'].to_numpy()

S = 0.5 * (a2 + a3) - a1                     # (Xyl+Man)/2 - Glu, log-space
eff = np.array([[S[M[:, i] == p].mean() - S.mean() for i in range(N_POS)]
              for p in range(1, N_PAR + 1)])
vlim = float(np.nanmax(np.abs(eff)))

# Shared with the structure figure (structure/make_structure_figure.py). Same hues, so a
# reader learns "blue = glucose-leaning, red = non-glucose-leaning" once and carries it
# between panels. The SCALES differ on purpose: this panel is whole-block effects (±2.05);
# the structure is each residue's share of one (±0.08, ~50x smaller, since a block effect
# is divided among its ~60 residues). Two colorbars, never one.
praxis_div = mcolors.LinearSegmentedColormap.from_list(
  'praxis_div', ['#1735f5', '#ffffff', '#dc1e04'])

def _ink(v):
  """Black or white cell text, from the actual background luminance rather than a
  magic saturation cutoff -- stays correct if the colormap ever changes."""
  r, g, b, _ = praxis_div((v + vlim) / (2 * vlim))
  lin = [c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4 for c in (r, g, b)]
  return 'white' if 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2] < 0.35 else 'black'

def draw(ax, minimal=False):
  im = ax.imshow(eff, aspect='auto', cmap=praxis_div, vmin=-vlim, vmax=vlim, origin='upper')
  ax.set_xticks(range(N_POS)); ax.set_yticks(range(N_PAR))
  if not minimal:
      ax.set_xticklabels(range(1, N_POS + 1)); ax.set_yticklabels(range(1, N_PAR + 1))
      ax.set_xlabel('Chimera block position'); ax.set_ylabel('Parental block identity')
      ax.set_title('Predicted non-glucose preference')
      for p in range(N_PAR):
          for i in range(N_POS):
              v = eff[p, i]
              ax.text(i, p, f'{v:+.1f}', ha='center', va='center', fontsize=9,
                      color=_ink(v))
  return im

# full (displayed)
fig, ax = plt.subplots(figsize=(8.2, 4.4))
im = draw(ax)
fig.tight_layout()
cb = fig.colorbar(im, ax=ax, shrink=0.9); cb.set_label('Effect on non-glucose preference')
fig.savefig('exports/marginal_effect_round24.png', dpi=300, bbox_inches='tight')
plt.show()

# minimal (saved to exports/, not displayed): same layout WITH colorbar, all text stripped
mfig = Figure(figsize=(8.2, 4.4)); max_ = mfig.add_subplot(111)
im_m = draw(max_, minimal=True)
max_.set_xticklabels([]); max_.set_yticklabels([])
mfig.tight_layout()
cbm = mfig.colorbar(im_m, ax=max_, shrink=0.9)
cbm.set_ticklabels([]); cbm.set_label('')
mfig.savefig('exports/marginal_effect_round24.svg', bbox_inches='tight')